# ChestX6 — Pipeline complet en cascade (Modèle 1 + Modèle 2)

Un seul notebook, **aucun prétraitement des données** : les `features` du parquet sont
envoyées telles quelles aux modèles (pas de resize, pas de rescale, pas de normalisation
des pixels).

```
chest_xray.parquet   ->  features (déjà prêtes) | label (nom de classe) | split
        |
        |  (aucune transformation — simple cast array -> Vector si nécessaire)
        v
MODÈLE 1 : StandardScaler -> PCA -> RandomForest        =>  0 = Sain / 1 = Malade
        |
        |  filter(prediction == 1.0 & class_name != "Normal")
        v
MODÈLE 2 : StandardScaler -> PCA -> OneVsRest(RandomForest)
        |
        v
Covid-19 / Emphysema / Pneumonia-Bacterial / Pneumonia-Viral / Tuberculosis
```

**Ce qui a changé par rapport aux 2 notebooks d'origine**

| Avant | Maintenant |
|---|---|
| UDF Python `resize_only` (48×48) exécutée dans **les 2** notebooks | supprimée — les features du parquet sont utilisées brutes |
| Modèle 1 sauvegardé puis **rechargé** depuis le disque par le notebook 2 | enchaîné **en mémoire** |
| Splits M2 écrits/relus en parquet (`m2_train/val/test`) | gardés en cache (plus de fichiers temporaires) |
| 2 sessions Spark, 2 fichiers | 1 session, 1 fichier |

> **Pré-requis** : `output/chest_xray.parquet` doit exister avec les colonnes
> `features` (array<float> ou vector), `label` (nom de la classe), `split` (train/val/test),
> et **toutes les images doivent avoir le même nombre de features** (vérifié à l'étape 2).

---
## 1. Initialisation

Chargement du `.env` (pour retrouver `PROJECT_ROOT`) et des libs Python. Pas de Spark
dans ce notebook : le parquet est déjà écrit sur disque par le pipeline Scala, on le lit
directement avec **pandas** et on entraîne les modèles avec **scikit-learn**.

In [ ]:
import os
import sys
import time
import math
import json
import shutil
import platform
import datetime
from pathlib import Path

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# --- Chargement du .env (pour PROJECT_ROOT / OUTPUT_PATH) ---
try:
    from dotenv import load_dotenv
    env_path = Path.cwd().parent / "src" / ".env"
    print(f"Chargement .env : {env_path} - existe : {env_path.exists()}")
    load_dotenv(dotenv_path=env_path, override=True)
except ImportError:
    print("WARN - python-dotenv non installe. Lance : pip install python-dotenv")

# --- Chemins ---
SCRIPT_DIR   = Path().resolve()
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", str(SCRIPT_DIR.parent)))
OUTPUT_DIR   = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams["font.family"] = "DejaVu Sans"

print(f"OK - Python       : {sys.version.split()[0]}")
print(f"OK - NumPy        : {np.__version__}")
print(f"OK - Pandas       : {pd.__version__}")
print(f"OK - PROJECT_ROOT : {PROJECT_ROOT}")
print(f"OK - OUTPUT_DIR   : {OUTPUT_DIR}")


---
## 2. Configuration des deux modèles

Tous les hyperparamètres sont regroupés ici. Rien d'autre à modifier dans le reste du notebook.

`PCA_MODE` :
- `"auto"` (recommandé) — la PCA est activée seulement si la dimension des features
  reste raisonnable (`PCA_MAX_DIM`). Spark calcule la PCA via une matrice de covariance
  `d × d` **sur le driver** : au-delà de ~10 000 features cela demande plusieurs Go de RAM,
  et Spark refuse carrément au-delà de 65 535.
- `"on"` — force la PCA. `"off"` — envoie les features standardisées directement à la forêt.

In [ ]:
# --- Données ---
PARQUET_PATH  = Path(os.environ.get("OUTPUT_PATH", str(PROJECT_ROOT / "results" / "preprocessed.parquet")))
HEALTHY_CLASS = "Normal"          # tout le reste = "Malade"

# --- PCA ---
PCA_MODE    = "auto"              # "auto" | "on" | "off"
PCA_MAX_DIM = 10000               # seuil de sécurité pour "auto"
PCA_K_M1    = 100                 # composantes principales — Modèle 1
PCA_K_M2    = 150                 # composantes principales — Modèle 2

# --- Modèle 1 : binaire Sain / Malade (RandomForest sklearn) ---
M1_NUM_TREES      = 250
M1_MAX_DEPTH      = 10
M1_MIN_SAMPLES    = 2             # equivalent minInstancesPerNode
M1_HEALTHY_WEIGHT = 4.0           # poids des images saines (class_weight)

# --- Modèle 2 : multi-classe (OneVsRest RandomForest) ---
M2_NUM_TREES      = 300
M2_MAX_DEPTH      = 12
M2_MIN_SAMPLES    = 1
M2_MAX_SAMPLES    = 0.9           # equivalent subsamplingRate

SEED   = 42
N_JOBS = -1                       # tous les coeurs CPU

# --- Sorties (joblib au lieu de Spark ML) ---
MODEL1_PATH  = str(OUTPUT_DIR / "chestx6_binary_model.joblib")
MODEL2_PATH  = str(OUTPUT_DIR / "chestx6_multiclass_model2.joblib")
INDEXER_PATH = str(OUTPUT_DIR / "chestx6_label_indexer.joblib")
JSON1_PATH   = str(OUTPUT_DIR / "resultats_test_modele1.json")
JSON2_PATH   = str(OUTPUT_DIR / "resultats_test_modele2.json")

print("OK - Configuration chargée")
print(f"  Parquet : {PARQUET_PATH}")
print(f"  M1 : RF({M1_NUM_TREES} arbres, depth={M1_MAX_DEPTH})")
print(f"  M2 : OneVsRest(RF {M2_NUM_TREES} arbres, depth={M2_MAX_DEPTH})")

---
## 3. Lecture du parquet — aucune transformation

Les pixels sont pris **tels quels**. La seule opération appliquée est un *cast* de type
`array<float>` → `Vector` (MLlib n'accepte que des `Vector` en entrée) : c'est un changement
de conteneur, **pas** une modification des valeurs. Si la colonne `features` est déjà un
`Vector`, même ce cast est sauté.

La cellule vérifie aussi que **toutes les images ont la même dimension** — c'est la condition
indispensable pour se passer du resize. Si ce n'est pas le cas, MLlib échouera à l'entraînement
et il faudra uniformiser la taille au moment du prétraitement (en amont, pas ici).

In [ ]:
if not PARQUET_PATH.exists():
    raise FileNotFoundError(f"Parquet introuvable : {PARQUET_PATH}")

print(f"Lecture : {PARQUET_PATH}")
# pandas lit un dossier de parts parquet directement (via pyarrow)
df_all = pd.read_parquet(PARQUET_PATH)
print(f"OK - {len(df_all)} lignes chargées")
print(f"Colonnes : {list(df_all.columns)}")
print(f"Types    : {df_all.dtypes.to_dict()}")

# --- Empiler les features en matrice 2D ---
# Le parquet Scala écrit la colonne "features" en array<float> :
# pyarrow le rend comme une colonne d'objets np.ndarray (1D).
# On empile pour obtenir un vrai (n_samples, n_features).
first = np.asarray(df_all["features"].iloc[0])
FEATURE_DIM = int(first.shape[0])

# Vérification : dimension unique ?
dims = df_all["features"].map(lambda a: len(a)).unique()
if len(dims) != 1:
    raise ValueError(
        f"Les images n'ont pas toutes la même dimension : {sorted(dims)}\n"
        "sklearn exige des vecteurs de taille identique. Uniformise en amont."
    )

side     = int(round(math.sqrt(FEATURE_DIM)))
IMG_SIZE = side if side * side == FEATURE_DIM else None

print(f"OK - Dimension uniforme : {FEATURE_DIM} features par image"
      + (f"  ({IMG_SIZE}x{IMG_SIZE})" if IMG_SIZE else "  (non carrée)"))

# --- Décision PCA ---
cov_mb = FEATURE_DIM * FEATURE_DIM * 8 / 1e6      # matrice de covariance d x d
if PCA_MODE == "auto":
    USE_PCA = FEATURE_DIM <= PCA_MAX_DIM
elif PCA_MODE == "on":
    USE_PCA = True
else:
    USE_PCA = False

if USE_PCA:
    print(f"OK - PCA activée (matrice de covariance ≈ {cov_mb:.0f} Mo)")
elif PCA_MODE == "off":
    print("OK - PCA désactivée (PCA_MODE='off').")
else:
    print(f"OK - PCA désactivée automatiquement : dim={FEATURE_DIM} > PCA_MAX_DIM={PCA_MAX_DIM}.")

---
## 4. Label binaire + splits train / val / test

`Normal` → 0 (Sain), tout le reste → 1 (Malade). Le nom réel de la classe est conservé dans
`class_name` : c'est lui qui servira de cible au Modèle 2. Les images saines reçoivent un poids
plus élevé dans le train pour compenser le déséquilibre.

In [ ]:
from sklearn.model_selection import train_test_split

def stack_features(sub_df):
    """Convertit la colonne 'features' (série d'arrays) en matrice 2D."""
    return np.vstack(sub_df["features"].values).astype(np.float32)

# --- Split simple : 80% train / 20% test (stratifie par classe) ---
print(f"Total : {len(df_all)} lignes")

df_pool = df_all.reset_index(drop=True)
df_train_pd, df_test_pd = train_test_split(
    df_pool, test_size=0.20, random_state=SEED, stratify=df_pool["label"]
)
df_train_pd = df_train_pd.reset_index(drop=True)
df_test_pd  = df_test_pd.reset_index(drop=True)

# --- Label binaire : Normal = 0 (Sain), reste = 1 (Malade) ---
for d in (df_train_pd, df_test_pd):
    d["class_name"]   = d["label"]
    d["binary_label"] = (d["label"] != HEALTHY_CLASS).astype(int)

# --- Matrices X + vecteurs y ---
X_train, y_train = stack_features(df_train_pd), df_train_pd["binary_label"].values
X_test,  y_test  = stack_features(df_test_pd),  df_test_pd["binary_label"].values

sample_weight_train = np.where(y_train == 0, M1_HEALTHY_WEIGHT, 1.0)

n_train, n_test = len(df_train_pd), len(df_test_pd)
print(f"OK - Train : {n_train} (80%) | Test : {n_test} (20%)")
print(f"    X_train shape : {X_train.shape}")
print(df_train_pd[["class_name", "binary_label"]].head(3))


---
## 5. Distribution des classes (train)

In [ ]:
dist_pd = (df_train_pd.groupby(["class_name", "binary_label"])
           .size().reset_index(name="count").sort_values("class_name"))
bin_pd = df_train_pd.groupby("binary_label").size().reset_index(name="count")
bin_pd["Categorie"] = bin_pd["binary_label"].map({0: "Sain (Normal)", 1: "Malade"})

print("Distribution TRAIN :")
print(dist_pd.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ["#2ecc71" if l == 0 else "#e74c3c" for l in dist_pd["binary_label"]]
axes[0].bar(dist_pd["class_name"], dist_pd["count"], color=colors)
axes[0].set_title("Distribution par classe (Train)")
axes[0].set_xlabel("Classe"); axes[0].set_ylabel("Nb images")
axes[0].tick_params(axis="x", rotation=30)
axes[1].pie(bin_pd["count"], labels=bin_pd["Categorie"],
            autopct="%1.1f%%", colors=["#2ecc71", "#e74c3c"], startangle=90)
axes[1].set_title("Sain vs Malade (Train)")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "distribution_classes.png"), dpi=100, bbox_inches="tight")
plt.show()
print("OK - Graphique sauvegardé dans output/")

---
---
# MODÈLE 1 — Sain vs Malade

## 6. Pipeline du Modèle 1

`StandardScaler` → `PCA` → `RandomForest`. Les features entrent **brutes** dans le scaler :
aucun redimensionnement n'a été appliqué en amont.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

steps_m1 = [("scaler", StandardScaler(with_mean=True, with_std=True))]

if USE_PCA:
    k1 = min(PCA_K_M1, FEATURE_DIM)
    steps_m1.append(("pca", PCA(n_components=k1, random_state=SEED)))

steps_m1.append((
    "rf",
    RandomForestClassifier(
        n_estimators=M1_NUM_TREES,
        max_depth=M1_MAX_DEPTH,
        min_samples_split=M1_MIN_SAMPLES,
        class_weight={0: M1_HEALTHY_WEIGHT, 1: 1.0},
        n_jobs=N_JOBS,
        random_state=SEED,
    )
))

pipeline_m1 = Pipeline(steps_m1)

desc_m1 = f"StandardScaler -> {'PCA(k=' + str(k1) + ') -> ' if USE_PCA else ''}RandomForest({M1_NUM_TREES} arbres)"
print(f"OK - Pipeline M1 : {desc_m1}")
print(f"     Entrée : vecteur {FEATURE_DIM} features (brutes, aucun prétraitement)")

## 7. Entraînement du Modèle 1

In [ ]:
print("Entraînement du Modèle 1 en cours...")
t0 = time.time()
# class_weight gère déjà le déséquilibre (sample_weight ferait double emploi)
model1 = pipeline_m1.fit(X_train, y_train)
elapsed_m1 = time.time() - t0
print(f"OK - Modèle 1 entraîné en {elapsed_m1:.1f}s ({elapsed_m1/60:.1f} min)")

## 8. Fonctions d'évaluation (binaire)

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix,
)

def evaluate_m1(model, X, y, split_name):
    y_pred  = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    acc  = accuracy_score(y, y_pred)
    auc  = roc_auc_score(y, y_proba)
    f1   = f1_score(y, y_pred, average="weighted")
    prec = precision_score(y, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y, y_pred, average="weighted", zero_division=0)
    print(f"\n{'='*45}\n  {split_name}\n{'='*45}")
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  AUC-ROC   : {auc:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Rappel    : {rec:.4f}")
    print(f"{'='*45}")
    return y_pred, dict(accuracy=acc, auc=auc, f1=f1, precision=prec, recall=rec)


def plot_cm_m1(y_true, y_pred, split_name, save_path=None):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im)
    lbs = ["Sain (0)", "Malade (1)"]
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(lbs); ax.set_yticklabels(lbs)
    ax.set_xlabel("Prédit"); ax.set_ylabel("Réel")
    ax.set_title(f"Matrice de confusion — {split_name}")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=16,
                    color="white" if cm[i, j] > cm.max()/2 else "black")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()

    print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn}")
    if (tp+fn) > 0: print(f"  Sensibilité (rappel malade) : {tp/(tp+fn):.4f}")
    if (tn+fp) > 0: print(f"  Spécificité (rappel sain)   : {tn/(tn+fp):.4f}")
    return cm

print("OK - Fonctions d'évaluation M1 définies")

## 9. Résultats du Modèle 1 (validation + test)

In [ ]:
y_test_pred_m1, metrics_test_m1 = evaluate_m1(model1, X_test, y_test, "M1 — TEST")
cm_test_m1 = plot_cm_m1(y_test, y_test_pred_m1, "Test",
                        str(OUTPUT_DIR / "confusion_matrix_test.png"))

## 10. Sauvegarde du Modèle 1

In [ ]:
import joblib

joblib.dump(model1, MODEL1_PATH, compress=3)
print(f"OK - Modèle 1 sauvegardé : {MODEL1_PATH}")

resultats_m1 = {
    "meta": {
        "modele"         : "Modele 1 - Binaire (Sain vs Malade)",
        "pipeline"       : desc_m1,
        "classes"        : ["Sain", "Malade"],
        "img_size"       : IMG_SIZE,
        "features_dim"   : FEATURE_DIM,
        "pretraitement"  : "aucun (features du parquet utilisees telles quelles)",
        "date_generation": datetime.datetime.now().isoformat(timespec="seconds"),
    },
    "metriques_globales": {
        "test": {k: round(v, 4) for k, v in metrics_test_m1.items()},
    },
    "matrice_confusion": {
        "labels": ["Sain", "Malade"],
        "test"  : cm_test_m1.tolist(),
    },
}
with open(JSON1_PATH, "w", encoding="utf-8") as f:
    json.dump(resultats_m1, f, indent=2, ensure_ascii=False)
print(f"OK - JSON M1 sauvegardé : {JSON1_PATH}")

---
---
# MODÈLE 2 — Identification de la maladie (cascade)

## 11. Application du Modèle 1 — en mémoire

Le Modèle 1 vient d'être entraîné : on l'applique directement aux 3 splits, **sans repasser
par le disque** (c'est ce que faisait `models2_mac.ipynb` en rechargeant le modèle et en
refaisant tout le resize).

On ne garde que `features`, `class_name`, `prediction` et `label` : les colonnes intermédiaires
(`scaled_features`, `pca_features`, `rawPrediction`…) doivent disparaître, sinon le pipeline du
Modèle 2 refusera de recréer des colonnes portant le même nom.

In [ ]:
def apply_m1(X, df_pd):
    y_hat = model1.predict(X)
    out = pd.DataFrame({
        "class_name": df_pd["class_name"].values,
        "label"     : df_pd["binary_label"].values,
        "prediction": y_hat,
    })
    return X, out

X_m1_train, df_m1_train = apply_m1(X_train, df_train_pd)
X_m1_test,  df_m1_test  = apply_m1(X_test,  df_test_pd)

print("OK - Prédictions du Modèle 1 appliquées :")
print(f"  Train : {len(df_m1_train)} images")
print(f"  Test  : {len(df_m1_test)} images")
print(df_m1_train.head(3))

## 12. Filtrage : garder uniquement les malades

On conserve les images que **M1 a prédites malades** et dont la **vraie classe n'est pas
`Normal`** : un sain mal classé par M1 n'a aucune maladie à apprendre à M2.

In [ ]:
def filter_malades(X, df_m1):
    mask = (df_m1["prediction"] == 1) & (df_m1["class_name"] != HEALTHY_CLASS)
    return X[mask.values], df_m1.loc[mask].reset_index(drop=True)

X_mal_train, df_mal_train = filter_malades(X_m1_train, df_m1_train)
X_mal_test,  df_mal_test  = filter_malades(X_m1_test,  df_m1_test)

print("Après filtrage (M1 prédit Malade + vraie classe != Normal) :")
print(f"  Train : {len(df_m1_train):>5} total → {len(df_mal_train):>5} gardés pour M2")
print(f"  Test  : {len(df_m1_test):>5} total → {len(df_mal_test):>5} gardés pour M2")
print()
print("Distribution par maladie dans le Train M2 :")
print(df_mal_train.groupby("class_name").size().reset_index(name="count").sort_values("class_name").to_string(index=False))

## 13. Encodage des labels (StringIndexer)

`Covid-19`, `Tuberculosis`… → indices numériques 0-4, triés par fréquence décroissante dans le
train. Le mapping est sauvegardé sur disque : l'application Streamlit en a besoin pour retraduire
une prédiction en nom de maladie.

In [ ]:
from sklearn.preprocessing import LabelEncoder

freq = df_mal_train["class_name"].value_counts()
CLASS_LABELS = freq.index.tolist()

label_encoder = LabelEncoder()
label_encoder.fit(CLASS_LABELS)
label_encoder.classes_ = np.array(CLASS_LABELS)

print("Mapping des classes (Modèle 2) :")
for i, cls in enumerate(CLASS_LABELS):
    print(f"  {i} → {cls}")

y_m2_train = label_encoder.transform(df_mal_train["class_name"].values)
y_m2_test  = label_encoder.transform(df_mal_test["class_name"].values)

print(f"\nOK - Train M2 : {len(y_m2_train)} | Test M2 : {len(y_m2_test)}")

## 14. Distribution des maladies (train M2)

In [ ]:
COLORS = {
    "Covid-19"           : "#e74c3c",
    "Emphysema"          : "#e67e22",
    "Pneumonia-Bacterial": "#9b59b6",
    "Pneumonia-Viral"    : "#3498db",
    "Tuberculosis"       : "#1abc9c",
}

dist_pd2 = (df_mal_train.groupby("class_name").size()
            .reset_index(name="count").sort_values("class_name"))
print("Distribution des maladies dans Train M2 :")
print(dist_pd2.to_string(index=False))

colors = [COLORS.get(c, "#95a5a6") for c in dist_pd2["class_name"]]
plt.figure(figsize=(9, 4))
bars = plt.bar(dist_pd2["class_name"], dist_pd2["count"],
               color=colors, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, dist_pd2["count"]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             str(val), ha="center", fontweight="bold", fontsize=10)
plt.title("Distribution des maladies — Train Modèle 2\n(images filtrées par le Modèle 1)", fontsize=12)
plt.xlabel("Maladie"); plt.ylabel("Nb images")
plt.tick_params(axis="x", rotation=30)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "distribution_maladies.png"), dpi=100, bbox_inches="tight")
plt.show()
print("OK - Graphique sauvegardé")

## 15. Pipeline du Modèle 2

`StandardScaler` → `PCA` → `OneVsRest(RandomForest)` : une forêt binaire par maladie.
Mêmes features brutes que le Modèle 1 — elles n'ont jamais été retouchées entre les deux modèles.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier

steps_m2 = [("scaler", StandardScaler(with_mean=True, with_std=True))]

if USE_PCA:
    k2 = min(PCA_K_M2, len(y_m2_train) - 1, FEATURE_DIM)
    steps_m2.append(("pca", PCA(n_components=k2, random_state=SEED)))

rf_base = RandomForestClassifier(
    n_estimators=M2_NUM_TREES,
    max_depth=M2_MAX_DEPTH,
    min_samples_split=M2_MIN_SAMPLES,
    max_features="sqrt",
    max_samples=M2_MAX_SAMPLES,   # equivalent subsamplingRate
    n_jobs=N_JOBS,
    random_state=SEED,
)
steps_m2.append(("ovr", OneVsRestClassifier(rf_base, n_jobs=N_JOBS)))

pipeline_m2 = Pipeline(steps_m2)

desc_m2 = (f"StandardScaler -> {'PCA(k=' + str(k2) + ') -> ' if USE_PCA else ''}"
           f"OneVsRest(RF {M2_NUM_TREES} arbres)")
print(f"OK - Pipeline M2 : {desc_m2}")
print(f"     Entrée : vecteur {FEATURE_DIM} features (brutes, aucun prétraitement)")
print(f"     Sortie : classe parmi {CLASS_LABELS}")

## 16. Entraînement du Modèle 2

In [ ]:
print("Entraînement du Modèle 2 en cours...")
t0 = time.time()
model2 = pipeline_m2.fit(X_mal_train, y_m2_train)
elapsed_m2 = time.time() - t0
print(f"OK - Modèle 2 entraîné en {elapsed_m2:.1f}s ({elapsed_m2/60:.1f} min)")

## 17. Évaluation du Modèle 2 + matrices de confusion

In [ ]:
def evaluate_m2(model, X, y, split_name):
    y_pred = model.predict(X)
    acc  = accuracy_score(y, y_pred)
    f1   = f1_score(y, y_pred, average="weighted")
    prec = precision_score(y, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y, y_pred, average="weighted", zero_division=0)
    print(f"\n{'='*45}\n  {split_name}\n{'='*45}")
    print(f"  Accuracy  : {acc:.4f} ({acc*100:.2f}%)")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Rappel    : {rec:.4f}")
    print(f"{'='*45}")
    return y_pred, dict(accuracy=acc, f1=f1, precision=prec, recall=rec)


def plot_cm_m2(y_true, y_pred, labels, split_name, save_path=None):
    n  = len(labels)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n)))

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=10)
    ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel("Prédit", fontsize=12)
    ax.set_ylabel("Réel",   fontsize=12)
    ax.set_title(f"Matrice de confusion — {split_name}", fontsize=13, fontweight="bold")
    threshold = cm.max() / 2 if cm.max() > 0 else 1
    for i in range(n):
        for j in range(n):
            val = cm[i, j]
            row_total = cm[i, :].sum()
            pct = val / row_total * 100 if row_total > 0 else 0
            ax.text(j, i, f"{val}\n({pct:.0f}%)", ha="center", va="center",
                    fontsize=8, fontweight="bold",
                    color="white" if val > threshold else "black")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()
    return cm


y_test_pred_m2, metrics_test_m2 = evaluate_m2(model2, X_mal_test, y_m2_test, "M2 — TEST")
cm_test_m2 = plot_cm_m2(y_m2_test, y_test_pred_m2, CLASS_LABELS, "Test",
                        str(OUTPUT_DIR / "confusion_matrix2_test.png"))

## 18. Export JSON pour Streamlit

Le fichier `resultats_test_modele2.json` garde **exactement le même schéma** qu'avant :
l'application Streamlit existante continue de fonctionner sans modification.

In [ ]:
def per_class(cm):
    out = {}
    for k, cls_name in enumerate(CLASS_LABELS):
        TP = int(cm[k, k])
        FP = int(cm[:, k].sum() - cm[k, k])
        FN = int(cm[k, :].sum() - cm[k, k])
        TN = int(cm.sum() - TP - FP - FN)
        precision = round(TP / (TP + FP), 4) if (TP + FP) > 0 else 0.0
        recall    = round(TP / (TP + FN), 4) if (TP + FN) > 0 else 0.0
        f1_cls    = round(2*precision*recall / (precision+recall), 4) if (precision+recall) > 0 else 0.0
        out[cls_name] = {
            "TP": TP, "FP": FP, "FN": FN, "TN": TN,
            "precision": precision, "recall": recall, "f1": f1_cls,
            "total_reel": int(cm[k, :].sum()),
        }
    return out

confusions = []
for i in range(len(CLASS_LABELS)):
    for j in range(len(CLASS_LABELS)):
        if i != j and cm_test_m2[i, j] > 0:
            confusions.append({
                "vrai"  : CLASS_LABELS[i],
                "predit": CLASS_LABELS[j],
                "count" : int(cm_test_m2[i, j]),
            })
confusions = sorted(confusions, key=lambda x: -x["count"])[:10]

dist_dict = df_mal_test.groupby("class_name").size().to_dict()
dist_dict = {k: int(v) for k, v in dist_dict.items()}

resultats_json = {
    "meta": {
        "modele"         : "Modele 2 - Multi-classe (cascade apres Modele 1)",
        "pipeline"       : desc_m2,
        "classes"        : CLASS_LABELS,
        "nb_classes"     : len(CLASS_LABELS),
        "total_test"     : int(cm_test_m2.sum()),
        "total_correct"  : int(np.trace(cm_test_m2)),
        "img_size"       : IMG_SIZE,
        "features_dim"   : FEATURE_DIM,
        "pretraitement"  : "aucun (features du parquet utilisees telles quelles)",
        "date_generation": datetime.datetime.now().isoformat(timespec="seconds"),
        "source_modele1" : MODEL1_PATH,
    },
    "metriques_globales": {
        "test": {
            "accuracy" : round(metrics_test_m2["accuracy"],  4),
            "f1"       : round(metrics_test_m2["f1"],        4),
            "precision": round(metrics_test_m2["precision"], 4),
            "recall"   : round(metrics_test_m2["recall"],    4),
        },
    },
    "metriques_par_classe": {
        "test": per_class(cm_test_m2),
    },
    "matrice_confusion": {
        "labels": CLASS_LABELS,
        "test"  : cm_test_m2.tolist(),
    },
    "distribution_test"    : dist_dict,
    "confusions_frequentes": confusions,
}

with open(JSON2_PATH, "w", encoding="utf-8") as f:
    json.dump(resultats_json, f, indent=2, ensure_ascii=False)

print(f"OK - JSON sauvegardé : {JSON2_PATH}")
print(f'     Accuracy (test) : {resultats_json["metriques_globales"]["test"]["accuracy"]}')
print(f'     F1-Score (test) : {resultats_json["metriques_globales"]["test"]["f1"]}')
print(f"     Classes         : {CLASS_LABELS}")

## 19. Sauvegarde du Modèle 2 + du mapping des classes

In [ ]:
joblib.dump(model2, MODEL2_PATH, compress=3)
print(f"OK - Modèle 2 sauvegardé : {MODEL2_PATH}")

joblib.dump(label_encoder, INDEXER_PATH)
print(f"OK - LabelEncoder sauvegardé : {INDEXER_PATH}")
print(f"     (labels : {CLASS_LABELS})")

---
---
## 20. Test de la cascade complète (M1 → M2)

On recharge les deux modèles **depuis le disque** — c'est exactement ce que fera l'app Streamlit —
et on les enchaîne sur quelques images de test : M1 décide sain/malade, et seules les images
prédites malades passent au M2 pour l'identification de la maladie.

In [ ]:
m1_loaded = joblib.load(MODEL1_PATH)
m2_loaded = joblib.load(MODEL2_PATH)
le_loaded = joblib.load(INDEXER_PATH)
print("OK - Modèles rechargés depuis le disque")

# Échantillon ALÉATOIRE (le parquet est souvent trié par classe)
N_DEMO = 20
rng = np.random.default_rng(SEED)
idx  = rng.choice(len(df_test_pd), size=min(N_DEMO, len(df_test_pd)), replace=False)

X_demo    = X_test[idx]
noms_reels = df_test_pd["class_name"].values[idx]

# --- Étape 1 : M1 sur toutes les images ---
pred_m1 = m1_loaded.predict(X_demo)

# --- Étape 2 : M2 uniquement sur celles que M1 a prédites malades ---
mask_malade = (pred_m1 == 1)
pred_m2 = np.full(len(X_demo), -1, dtype=int)
if mask_malade.any():
    pred_m2[mask_malade] = m2_loaded.predict(X_demo[mask_malade])

print(f"\n{'':<4}{'Vraie classe':<22}{'M1':<10}{'Diagnostic final'}")
print("-" * 64)
n_ok = 0
for i in range(len(X_demo)):
    vraie = noms_reels[i]
    if pred_m1[i] == 0:
        m1_txt, final = "Sain", "Sain (Normal)"
        correct = (vraie == HEALTHY_CLASS)
    else:
        m1_txt = "Malade"
        final  = le_loaded.classes_[pred_m2[i]] if pred_m2[i] != -1 else "?"
        correct = (final == vraie)
    n_ok += int(correct)
    print(f"{'OK ' if correct else 'XX ':<4}{vraie:<22}{m1_txt:<10}{final}")
print("-" * 64)
print(f"Cascade correcte : {n_ok}/{len(X_demo)}")

---
## 21. Résumé final

In [ ]:
print("=" * 60)
print("  RÉSUMÉ — PIPELINE EN CASCADE (scikit-learn)")
print("=" * 60)
print(f"  Prétraitement : AUCUN — features du parquet telles quelles")
print(f"  Dimension     : {FEATURE_DIM} features/image"
      + (f" ({IMG_SIZE}x{IMG_SIZE})" if IMG_SIZE else ""))
print(f"  Données       : train={n_train} (80%)  test={n_test} (20%)")
print()
print(f"  [MODÈLE 1] {desc_m1}")
print(f"    Entraînement : {elapsed_m1:.1f}s")
print(f"    TEST  Accuracy={metrics_test_m1['accuracy']:.4f}  AUC={metrics_test_m1['auc']:.4f}  F1={metrics_test_m1['f1']:.4f}")
print()
print(f"  [MODÈLE 2] {desc_m2}")
print(f"    Classes      : {', '.join(CLASS_LABELS)}")
print(f"    Entraînement : {elapsed_m2:.1f}s")
print(f"    TEST  Accuracy={metrics_test_m2['accuracy']:.4f}  F1={metrics_test_m2['f1']:.4f}")
print()
print("  Fichiers générés dans output/ :")
for f in ["chestx6_binary_model.joblib", "chestx6_multiclass_model2.joblib",
          "chestx6_label_indexer.joblib",
          "resultats_test_modele1.json", "resultats_test_modele2.json",
          "distribution_classes.png", "distribution_maladies.png",
          "confusion_matrix_test.png", "confusion_matrix2_test.png"]:
    print(f"    - {f}")
print("=" * 60)

In [ ]:
print("OK - Notebook termine")